In [29]:
import os
import platform
import pandas as pd
import numpy as np
import torch
import pytorch_lightning as pl

from pytorch_lightning.callbacks.early_stopping import EarlyStopping
from torch.utils.data import DataLoader
from multiprocessing import cpu_count

In [30]:
seed = 42
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
pl.seed_everything(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

Global seed set to 42


In [31]:
os.getcwd()

'/home/czy/KT/BRIKT_mine'

In [32]:
# SEQ_LEN = 50 # DKT, SAKT
# SEQ_LEN = 200 # DKVMN
SEQ_LEN = 100 # DKVMN
BATCH_SIZE = 64
EMBED_DIM = 128
NUM_WORKERS = 0 if platform.system() == 'Windows' else cpu_count()
print("os:{}, num-workers:{}".format(platform.system(), NUM_WORKERS))

os:Linux, num-workers:16


In [33]:
# dkt, and sakt, and dkvmn...
# model = 'dkt' # 模型为dkt
model = 'dkvmn' # 模型为dkvmn
# model = 'sakt' # 模型为dkvmn
q_is_s = True # skill作为输入

In [34]:
key_q = 'q_idx'
key_s = 's_idx'


In [35]:
with open('dataset/assist09/pro_id_dict.txt', 'r') as f:
    pro_id_dict = eval(f.read())
N_QUESTION = len(pro_id_dict)

with open('dataset/assist09/skill_id_dict.txt', 'r') as f:
    skill_id_dict = eval(f.read())
N_SKILL = len(skill_id_dict)
N_QUESTION, N_SKILL

(16891, 101)

In [36]:
# train : validation = 80% : 20%
df_train = pd.read_csv('dataset/assist09/train.csv', low_memory=False, encoding="ISO-8859-1")
df_val = pd.read_csv('dataset/assist09/test.csv', low_memory=False, encoding="ISO-8859-1")
df_train.head()

,user_id,q_idx,s_idx,q_type,q_diff,ms_first_response,attempt_count,correct
0,96245.0,11.0,0.0,0.0,0.625000,0.010081,0.000523,0.0
1,96245.0,24.0,0.0,0.0,0.738095,0.010259,0.000523,0.0
2,96245.0,31.0,0.0,0.0,0.613636,0.010298,0.000523,0.0
3,96245.0,48.0,0.0,0.0,0.787879,0.010179,0.000523,0.0
4,96245.0,56.0,0.0,0.0,0.736842,0.010344,0.000523,0.0


In [37]:
KEY = key_s if q_is_s else key_q
NUM_Q_OR_S = N_SKILL if q_is_s else N_QUESTION

In [38]:
def generate_group_by_df(df):
    group = df[['user_id', KEY, 'correct']].groupby(['user_id']).apply(lambda r: (
            r[KEY].values,
            r['correct'].values
            ))
    return group

In [39]:
train = generate_group_by_df(df_train) 
val = generate_group_by_df(df_val)
train.iloc[0] # 第一个user的s_id, correct

(array([ 1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  9.,
         9.,  9.,  9., 10., 10., 10.]),
 array([0., 1., 0., 0., 0., 0., 0., 0., 1., 0., 1., 0., 0., 0., 0., 0., 1.,
        1., 1.]))

In [40]:
from data_loader.dktdataset import DKTDataset

train_dataset = DKTDataset(train, N_SKILL, SEQ_LEN)
train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)

val_dataset = DKTDataset(val, N_SKILL, SEQ_LEN)
val_dataloader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

print("train:{}, test:{}".format(len(train_dataset), len(val_dataset)))

train:3320, test:831


In [41]:
from model.dkt import DKTModule
from model.sakt import SAKTModule
from model.dkvmn import DKVMNModule
import warnings
warnings.filterwarnings('ignore')
if model == 'dkt':
    model = DKTModule(n_question=NUM_Q_OR_S)
elif model == 'sakt':
    model = SAKTModule(n_question=NUM_Q_OR_S, max_seq=SEQ_LEN, embed_dim=EMBED_DIM)
elif model == 'dkvmn':
    model = DKVMNModule(n_question=NUM_Q_OR_S)

print("num of question:{}, num of skill:{}".format(N_QUESTION, N_SKILL))
print("question is skill：{}, num_q_or_s:{}".format(q_is_s, NUM_Q_OR_S))
print("model:{}".format(model))

num of question:16891, num of skill:101
question is skill：True, num_q_or_s:101
model:DKVMNModule(
  (loss): BCEWithLogitsLoss()
  (model): DKVMNMODEL(
    (read_embed_linear): Linear(in_features=150, out_features=10, bias=True)
    (predict_linear): Linear(in_features=10, out_features=1, bias=True)
    (mem): DKVMN(
      (key_head): DKVMNHeadGroup()
      (value_head): DKVMNHeadGroup(
        (erase): Linear(in_features=100, out_features=100, bias=True)
        (add): Linear(in_features=100, out_features=100, bias=True)
      )
    )
    (q_embed): Embedding(102, 50, padding_idx=0)
    (qa_embed): Embedding(203, 100, padding_idx=0)
  )
)


In [42]:
checkpoint_callback = pl.callbacks.ModelCheckpoint(save_top_k=1, verbose=True, monitor='v_auc', mode='max')

# sakt.train_dataloader
trainer = pl.Trainer(
    gpus=1, 
    max_epochs=200, 
    auto_lr_find=True, 
    callbacks=[checkpoint_callback, EarlyStopping(monitor="v_auc", mode="max", patience=6)]
)

trainer.fit(model=model, train_dataloaders=train_dataloader,val_dataloaders=val_dataloader)

GPU available: True, used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name  | Type              | Params
--------------------------------------------
0 | loss  | BCEWithLogitsLoss | 0     
1 | model | DKVMNMODEL        | 50.1 K
--------------------------------------------
50.1 K    Trainable params
0         Non-trainable params
50.1 K    Total params
0.200     Total estimated model params size (MB)


Sanity Checking: 0it [00:00, ?it/s]

Training: 0it [00:00, ?it/s]

RuntimeError: CUDA out of memory. Tried to allocate 2.00 MiB (GPU 0; 10.76 GiB total capacity; 8.94 GiB already allocated; 5.69 MiB free; 9.18 GiB reserved in total by PyTorch) If reserved memory is >> allocated memory try setting max_split_size_mb to avoid fragmentation.  See documentation for Memory Management and PYTORCH_CUDA_ALLOC_CONF

: 